# Software FJ: Sistema Integral de Gestión de Clientes, Servicios y Reservas

**Universidad Nacional Abierta y a Distancia (UNAD) · Programación (213023) · Fase 4 – Componente práctico**

Sistema orientado a objetos, sin base de datos, que gestiona clientes, servicios y reservas aplicando abstracción, herencia, polimorfismo, encapsulación y manejo avanzado de excepciones. Todos los errores y eventos se registran en el archivo `software_fj.log`.

**Estructura del notebook**
1. Registro de eventos (logs)
2. Excepciones personalizadas
3. Clase abstracta `Entidad` y clase `Cliente`
4. Clase abstracta `Servicio` y tres servicios especializados
5. Clase `Reserva`
6. Clase `GestorSistema`
7. Simulación de 18 operaciones (válidas e inválidas)

## 1. Registro de eventos (logs)

In [9]:
from abc import ABC, abstractmethod   # Herramientas para crear clases abstractas
from datetime import datetime         # Para poner fecha y hora a cada registro
import re                             # Expresiones regulares para validar correos

ARCHIVO_LOG = "software_fj.log"       # Archivo donde se guardan errores y eventos


def registrar_log(nivel, mensaje):
    """Escribe una línea en el archivo de logs y la muestra en pantalla."""
    # Se arma la línea con fecha, nivel (INFO, ERROR...) y mensaje
    linea = f"{datetime.now():%Y-%m-%d %H:%M:%S} [{nivel}] {mensaje}"
    try:
        # 'with' garantiza que el archivo se cierre aunque ocurra un error
        with open(ARCHIVO_LOG, "a", encoding="utf-8") as archivo:
            archivo.write(linea + "\n")
    except OSError as error:
        # Si el log falla, el programa NO debe caerse: solo avisa
        print(f"[ADVERTENCIA] No se pudo escribir en el log: {error}")
    print(linea)

## 2. Excepciones personalizadas
Todas heredan de `SoftwareFJError`, así el sistema puede capturar cualquier error propio con un solo `except`.

In [10]:
class SoftwareFJError(Exception):
    """Excepción base de todo el sistema."""

class ClienteInvalidoError(SoftwareFJError):
    """Datos del cliente inválidos (nombre, correo, teléfono...)."""

class ParametroFaltanteError(SoftwareFJError):
    """Falta un parámetro obligatorio."""

class ServicioInvalidoError(SoftwareFJError):
    """Datos del servicio inválidos (tarifa, nombre, parámetros)."""

class ServicioNoDisponibleError(SoftwareFJError):
    """El servicio solicitado no está disponible."""

class OperacionNoPermitidaError(SoftwareFJError):
    """La operación no es válida en el estado actual."""

class ReservaError(SoftwareFJError):
    """Error general al procesar una reserva."""

class CalculoInconsistenteError(SoftwareFJError):
    """El cálculo de costos produjo un valor inconsistente."""

## 3. Clase abstracta `Entidad` y clase `Cliente`
`Entidad` representa cualquier objeto con identificador. `Cliente` la hereda y **encapsula** sus datos personales (atributos privados con validación en los *setters*).

In [11]:
class Entidad(ABC):
    """Clase abstracta: representa entidades generales del sistema."""

    def __init__(self, identificador):
        # Se valida que el identificador exista y sea texto
        if not identificador or not isinstance(identificador, str):
            raise ParametroFaltanteError("El identificador es obligatorio (texto).")
        self._identificador = identificador   # Atributo protegido

    @property
    def identificador(self):
        return self._identificador

    @abstractmethod
    def describir(self):
        """Cada subclase debe describirse a sí misma."""


class Cliente(Entidad):
    """Cliente de Software FJ con datos personales encapsulados."""

    def __init__(self, identificador, nombre, correo, telefono):
        super().__init__(identificador)
        # Al asignar, se ejecutan los setters que validan cada dato
        self.nombre = nombre
        self.correo = correo
        self.telefono = telefono

    @property
    def nombre(self):
        return self.__nombre

    @nombre.setter
    def nombre(self, valor):
        # Solo letras y espacios, mínimo 3 caracteres
        if not valor or not isinstance(valor, str):
            raise ParametroFaltanteError("El nombre del cliente es obligatorio.")
        if len(valor.strip()) < 3 or not all(c.isalpha() or c.isspace() for c in valor):
            raise ClienteInvalidoError(f"Nombre inválido: '{valor}'.")
        self.__nombre = valor.strip().title()

    @property
    def correo(self):
        return self.__correo

    @correo.setter
    def correo(self, valor):
        # Formato básico usuario@dominio.ext
        if not valor or not isinstance(valor, str):
            raise ParametroFaltanteError("El correo del cliente es obligatorio.")
        if not re.fullmatch(r"[\w.+-]+@[\w-]+\.[\w.-]+", valor):
            raise ClienteInvalidoError(f"Correo inválido: '{valor}'.")
        self.__correo = valor.lower()

    @property
    def telefono(self):
        return self.__telefono

    @telefono.setter
    def telefono(self, valor):
        # Solo dígitos, entre 7 y 10
        if not valor or not isinstance(valor, str):
            raise ParametroFaltanteError("El teléfono del cliente es obligatorio.")
        if not valor.isdigit() or not 7 <= len(valor) <= 10:
            raise ClienteInvalidoError(f"Teléfono inválido: '{valor}'.")
        self.__telefono = valor

    def describir(self):
        return f"Cliente {self.identificador}: {self.nombre} <{self.correo}> Tel. {self.telefono}"

## 4. Clase abstracta `Servicio` y servicios especializados
`Servicio` define el contrato (métodos abstractos). Cada servicio hijo **sobrescribe** los métodos (polimorfismo). El método `calcular_costo` simula la **sobrecarga** de Python con parámetros opcionales (impuesto y descuento).

In [12]:
class Servicio(Entidad, ABC):
    """Clase abstracta de la que heredan todos los servicios."""

    IVA = 0.19   # Impuesto por defecto

    def __init__(self, identificador, nombre, tarifa_hora, disponible=True):
        super().__init__(identificador)
        if not nombre or not isinstance(nombre, str):
            raise ParametroFaltanteError("El nombre del servicio es obligatorio.")
        # La tarifa debe ser un número positivo
        if not isinstance(tarifa_hora, (int, float)) or tarifa_hora <= 0:
            raise ServicioInvalidoError(f"Tarifa inválida: {tarifa_hora}.")
        self.nombre = nombre
        self._tarifa_hora = float(tarifa_hora)
        self.disponible = disponible

    @property
    def tarifa_hora(self):
        return self._tarifa_hora

    def validar_parametros(self, horas):
        """Validación común: las horas deben ser un número entre 0 y 24."""
        if horas is None:
            raise ParametroFaltanteError("Las horas son obligatorias.")
        if not isinstance(horas, (int, float)) or not 0 < horas <= 24:
            raise ServicioInvalidoError(f"Duración inválida: {horas} horas.")

    @abstractmethod
    def costo_base(self, horas):
        """Cada servicio define cómo calcula su costo antes de impuestos."""

    def calcular_costo(self, horas, impuesto=None, descuento=0.0):
        """
        Método 'sobrecargado' (con parámetros opcionales):
          calcular_costo(2)                  -> con IVA por defecto
          calcular_costo(2, 0.10)            -> con otro impuesto
          calcular_costo(2, 0.19, 0.05)      -> con impuesto y descuento
        """
        self.validar_parametros(horas)
        if impuesto is None:
            impuesto = self.IVA
        if not 0 <= impuesto <= 1 or not 0 <= descuento < 1:
            raise CalculoInconsistenteError("Impuesto o descuento fuera de rango.")
        base = self.costo_base(horas)
        total = base * (1 - descuento) * (1 + impuesto)
        if total <= 0:
            raise CalculoInconsistenteError(f"Costo inconsistente: {total}.")
        return round(total, 2)


class ReservaSala(Servicio):
    """Reserva de salas: descuento automático por reservas largas."""

    def __init__(self, identificador, nombre, tarifa_hora, capacidad, disponible=True):
        super().__init__(identificador, nombre, tarifa_hora, disponible)
        if not isinstance(capacidad, int) or capacidad <= 0:
            raise ServicioInvalidoError(f"Capacidad inválida: {capacidad}.")
        self.capacidad = capacidad

    def costo_base(self, horas):
        base = self.tarifa_hora * horas
        # 10% de descuento si se reservan 4 horas o más
        return base * 0.90 if horas >= 4 else base

    def describir(self):
        return f"Sala '{self.nombre}' (capacidad {self.capacidad}) a ${self.tarifa_hora:,.0f}/hora"


class AlquilerEquipo(Servicio):
    """Alquiler de equipos: se suma un seguro fijo."""

    SEGURO = 15000

    def __init__(self, identificador, nombre, tarifa_hora, disponible=True):
        super().__init__(identificador, nombre, tarifa_hora, disponible)

    def costo_base(self, horas):
        return self.tarifa_hora * horas + self.SEGURO

    def describir(self):
        return f"Equipo '{self.nombre}' a ${self.tarifa_hora:,.0f}/hora + seguro de ${self.SEGURO:,}"


class AsesoriaEspecializada(Servicio):
    """Asesorías: el valor depende del nivel del asesor."""

    NIVELES = {"junior": 1.0, "senior": 1.5, "experto": 2.0}

    def __init__(self, identificador, nombre, tarifa_hora, nivel, disponible=True):
        super().__init__(identificador, nombre, tarifa_hora, disponible)
        if nivel not in self.NIVELES:
            raise ServicioInvalidoError(f"Nivel inválido: '{nivel}'. Use {list(self.NIVELES)}.")
        self.nivel = nivel

    def costo_base(self, horas):
        return self.tarifa_hora * horas * self.NIVELES[self.nivel]

    def describir(self):
        return f"Asesoría '{self.nombre}' nivel {self.nivel} a ${self.tarifa_hora:,.0f}/hora"

## 5. Clase `Reserva`
Integra cliente, servicio, duración y estado. Usa `try/except/else/finally` y **encadenamiento de excepciones** (`raise ... from ...`).

In [13]:
class Reserva:
    """Reserva de un servicio por parte de un cliente."""

    def __init__(self, identificador, cliente, servicio, horas):
        # Validaciones: no se permiten datos faltantes
        if cliente is None or servicio is None:
            raise ParametroFaltanteError("La reserva requiere cliente y servicio.")
        if not isinstance(cliente, Cliente) or not isinstance(servicio, Servicio):
            raise ReservaError("Cliente o servicio con tipo incorrecto.")
        servicio.validar_parametros(horas)   # Reutiliza la validación del servicio
        self.identificador = identificador
        self.cliente = cliente
        self.servicio = servicio
        self.horas = horas
        self.estado = "PENDIENTE"            # PENDIENTE -> CONFIRMADA o CANCELADA
        self.costo = None

    def confirmar(self, impuesto=None, descuento=0.0):
        """Confirma la reserva y calcula su costo."""
        try:
            if self.estado != "PENDIENTE":
                raise OperacionNoPermitidaError(
                    f"No se puede confirmar una reserva en estado {self.estado}.")
            if not self.servicio.disponible:
                raise ServicioNoDisponibleError(
                    f"El servicio '{self.servicio.nombre}' no está disponible.")
            costo = self.servicio.calcular_costo(self.horas, impuesto, descuento)
        except ServicioNoDisponibleError as error:
            # Encadenamiento: se envuelve el error original en uno más general
            raise ReservaError(f"Reserva {self.identificador} no confirmada") from error
        except CalculoInconsistenteError as error:
            raise ReservaError(f"Reserva {self.identificador}: fallo en el cálculo") from error
        else:
            # Solo se ejecuta si NO hubo excepciones
            self.estado = "CONFIRMADA"
            self.costo = costo
            return costo
        finally:
            # Se ejecuta siempre, haya error o no
            registrar_log("INFO", f"Intento de confirmación de la reserva {self.identificador}")

    def cancelar(self):
        """Cancela la reserva si aún es posible."""
        if self.estado == "CANCELADA":
            raise OperacionNoPermitidaError(f"La reserva {self.identificador} ya está cancelada.")
        self.estado = "CANCELADA"
        self.costo = None

    def procesar(self):
        """Confirma la reserva y devuelve un resumen."""
        costo = self.confirmar()
        return (f"Reserva {self.identificador}: {self.cliente.nombre} - "
                f"{self.servicio.nombre} - {self.horas} h - ${costo:,.2f}")

    def __str__(self):
        return (f"Reserva {self.identificador} [{self.estado}] {self.cliente.nombre} / "
                f"{self.servicio.nombre} / {self.horas} h")

## 6. Clase `GestorSistema`
Administra las listas internas (clientes, servicios, reservas). El método `ejecutar` captura cualquier error, lo registra en el log y **mantiene el sistema activo**.

In [14]:
class GestorSistema:
    """Coordina clientes, servicios y reservas usando listas internas."""

    def __init__(self):
        self.clientes = []
        self.servicios = []
        self.reservas = []

    def ejecutar(self, descripcion, funcion, *args, **kwargs):
        """Ejecuta una operación sin permitir que un error detenga el programa."""
        try:
            resultado = funcion(*args, **kwargs)
        except SoftwareFJError as error:
            # Error previsto del sistema (se incluye la causa si hay encadenamiento)
            causa = f" | causa: {error.__cause__}" if error.__cause__ else ""
            registrar_log("ERROR", f"{descripcion} -> {type(error).__name__}: {error}{causa}")
            return None
        except Exception as error:
            # Error inesperado: también se registra y el sistema sigue
            registrar_log("CRITICO", f"{descripcion} -> Error inesperado {type(error).__name__}: {error}")
            return None
        else:
            registrar_log("INFO", f"{descripcion} -> OK")
            return resultado

    def registrar_cliente(self, identificador, nombre, correo, telefono):
        if any(c.identificador == identificador for c in self.clientes):
            raise ClienteInvalidoError(f"Ya existe un cliente con id {identificador}.")
        cliente = Cliente(identificador, nombre, correo, telefono)
        self.clientes.append(cliente)
        return cliente

    def registrar_servicio(self, servicio):
        if any(s.identificador == servicio.identificador for s in self.servicios):
            raise ServicioInvalidoError(f"Ya existe un servicio con id {servicio.identificador}.")
        self.servicios.append(servicio)
        return servicio

    def buscar(self, lista, identificador):
        for elemento in lista:
            if elemento.identificador == identificador:
                return elemento
        raise ParametroFaltanteError(f"No se encontró el id '{identificador}'.")

    def crear_reserva(self, id_reserva, id_cliente, id_servicio, horas):
        cliente = self.buscar(self.clientes, id_cliente)
        servicio = self.buscar(self.servicios, id_servicio)
        reserva = Reserva(id_reserva, cliente, servicio, horas)
        self.reservas.append(reserva)
        return reserva

    def resumen(self):
        confirmadas = [r for r in self.reservas if r.estado == "CONFIRMADA"]
        total = sum(r.costo for r in confirmadas)
        print("\n===== RESUMEN FINAL =====")
        print(f"Clientes registrados : {len(self.clientes)}")
        print(f"Servicios creados    : {len(self.servicios)}")
        print(f"Reservas totales     : {len(self.reservas)}")
        print(f"Reservas confirmadas : {len(confirmadas)}  (ingresos: ${total:,.2f})")
        for r in self.reservas:
            print(" -", r)

## 7. Simulación de 18 operaciones
Incluye registros válidos e inválidos de clientes, servicios correctos e incorrectos, y reservas exitosas y fallidas. El programa **nunca se detiene**.

In [15]:
# Se reinicia el log para que cada ejecución quede limpia
open(ARCHIVO_LOG, "w", encoding="utf-8").close()

sistema = GestorSistema()
ej = sistema.ejecutar   # Atajo para escribir menos

# --- Clientes ---
ej("Op 1: cliente válido (Ana)", sistema.registrar_cliente, "C1", "Ana Gómez", "ana@correo.com", "3001234567")
ej("Op 2: cliente con correo inválido", sistema.registrar_cliente, "C2", "Luis Pérez", "luis-correo.com", "3109876543")
ej("Op 3: cliente con teléfono inválido", sistema.registrar_cliente, "C3", "Marta Ruiz", "marta@correo.com", "abc123")
ej("Op 4: cliente válido (Carlos)", sistema.registrar_cliente, "C4", "Carlos Díaz", "carlos@correo.com", "3201112233")

# --- Servicios ---
ej("Op 5: sala válida", sistema.registrar_servicio, ReservaSala("S1", "Sala Ejecutiva", 80000, 10))
ej("Op 6: equipo válido", sistema.registrar_servicio, AlquilerEquipo("S2", "Proyector HD", 25000))
ej("Op 7: asesoría con nivel inválido", lambda: sistema.registrar_servicio(
    AsesoriaEspecializada("S3", "Asesoría en Python", 120000, "maestro")))
ej("Op 8: asesoría válida", lambda: sistema.registrar_servicio(
    AsesoriaEspecializada("S4", "Asesoría en Python", 120000, "senior")))

# --- Reservas ---
r1 = ej("Op 9: reserva sala 5 h", sistema.crear_reserva, "R1", "C1", "S1", 5)
ej("Op 10: confirmar reserva R1", r1.procesar)
ej("Op 11: reserva con duración inválida (30 h)", sistema.crear_reserva, "R2", "C4", "S2", 30)
ej("Op 12: reserva con cliente inexistente", sistema.crear_reserva, "R3", "C99", "S1", 2)

# --- Operaciones extra (servicio no disponible, cancelación, doble cancelación) ---
sistema.servicios[1].disponible = False   # El proyector deja de estar disponible
r4 = ej("Op 13: reserva de equipo no disponible", sistema.crear_reserva, "R4", "C4", "S2", 3)
ej("Op 14: confirmar reserva R4 (debe fallar)", r4.confirmar)
r5 = ej("Op 15: reserva asesoría 2 h", sistema.crear_reserva, "R5", "C4", "S4", 2)
ej("Op 16: confirmar R5", r5.procesar)
ej("Op 17: cancelar R5", r5.cancelar)
ej("Op 18: cancelar R5 otra vez (debe fallar)", r5.cancelar)

sistema.resumen()

2026-11-17 18:55:57 [INFO] Op 1: cliente válido (Ana) -> OK
2026-11-17 18:55:57 [ERROR] Op 2: cliente con correo inválido -> ClienteInvalidoError: Correo inválido: 'luis-correo.com'.
2026-11-17 18:55:57 [ERROR] Op 3: cliente con teléfono inválido -> ClienteInvalidoError: Teléfono inválido: 'abc123'.
2026-11-17 18:55:57 [INFO] Op 4: cliente válido (Carlos) -> OK
2026-11-17 18:55:57 [INFO] Op 5: sala válida -> OK
2026-11-17 18:55:57 [INFO] Op 6: equipo válido -> OK
2026-11-17 18:55:57 [ERROR] Op 7: asesoría con nivel inválido -> ServicioInvalidoError: Nivel inválido: 'maestro'. Use ['junior', 'senior', 'experto'].
2026-11-17 18:55:57 [INFO] Op 8: asesoría válida -> OK
2026-11-17 18:55:57 [INFO] Op 9: reserva sala 5 h -> OK
2026-11-17 18:55:57 [INFO] Intento de confirmación de la reserva R1
2026-11-17 18:55:57 [INFO] Op 10: confirmar reserva R1 -> OK
2026-11-17 18:55:57 [ERROR] Op 11: reserva con duración inválida (30 h) -> ServicioInvalidoError: Duración inválida: 30 horas.
2026-11-17 18

### Contenido del archivo de logs

In [16]:
# Se lee el log de forma segura y se muestra en pantalla
try:
    with open(ARCHIVO_LOG, "r", encoding="utf-8") as archivo:
        print(archivo.read())
except FileNotFoundError:
    print("El archivo de logs aún no existe.")

2026-11-17 18:55:57 [INFO] Op 1: cliente válido (Ana) -> OK
2026-11-17 18:55:57 [ERROR] Op 2: cliente con correo inválido -> ClienteInvalidoError: Correo inválido: 'luis-correo.com'.
2026-11-17 18:55:57 [ERROR] Op 3: cliente con teléfono inválido -> ClienteInvalidoError: Teléfono inválido: 'abc123'.
2026-11-17 18:55:57 [INFO] Op 4: cliente válido (Carlos) -> OK
2026-11-17 18:55:57 [INFO] Op 5: sala válida -> OK
2026-11-17 18:55:57 [INFO] Op 6: equipo válido -> OK
2026-11-17 18:55:57 [ERROR] Op 7: asesoría con nivel inválido -> ServicioInvalidoError: Nivel inválido: 'maestro'. Use ['junior', 'senior', 'experto'].
2026-11-17 18:55:57 [INFO] Op 8: asesoría válida -> OK
2026-11-17 18:55:57 [INFO] Op 9: reserva sala 5 h -> OK
2026-11-17 18:55:57 [INFO] Intento de confirmación de la reserva R1
2026-11-17 18:55:57 [INFO] Op 10: confirmar reserva R1 -> OK
2026-11-17 18:55:57 [ERROR] Op 11: reserva con duración inválida (30 h) -> ServicioInvalidoError: Duración inválida: 30 horas.
2026-11-17 18

## Cómo se cumplen los requisitos de la actividad

| Requisito | Dónde se ve |
|---|---|
| Clase abstracta de entidades | `Entidad` |
| Clase `Cliente` con validaciones y encapsulación | `Cliente` (atributos privados + *setters* que validan) |
| Clase abstracta `Servicio` y 3 servicios | `Servicio`, `ReservaSala`, `AlquilerEquipo`, `AsesoriaEspecializada` |
| Polimorfismo y métodos sobrescritos | `costo_base` y `describir` en cada servicio |
| Clase `Reserva` (confirmar, cancelar, procesar) | `Reserva` |
| Métodos sobrecargados | `calcular_costo(horas, impuesto, descuento)` con parámetros opcionales |
| try/except, else, finally y encadenamiento | `Reserva.confirmar`, `GestorSistema.ejecutar` |
| Excepciones personalizadas | Sección 2 |
| Archivo de logs | `registrar_log` → `software_fj.log` |
| Al menos 10 operaciones | Sección 7 (18 operaciones) |